<a href="https://colab.research.google.com/github/DrMuratAltun/meb-yegitek-derin-ogrenme/blob/main/04_cnn_giysi_siniflandirma.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Colab'da Aç"/></a> &nbsp; <a href="https://github.com/DrMuratAltun/meb-yegitek-derin-ogrenme/blob/main/04_cnn_giysi_siniflandirma.ipynb" target="_parent"><img src="https://img.shields.io/badge/GitHub-Kaynak-181717?logo=github" alt="GitHub'da Görüntüle"/></a>

> 🚀 **Bu notebook'u tarayıcıda çalıştırmak için sol üstteki "Colab'da Aç" butonuna tıklayın** — hiçbir şey kurmanıza gerek yok!


<div style="background: linear-gradient(135deg, #0B3D91 0%, #1565C0 60%, #1E88E5 100%); padding: 26px 30px; border-radius: 14px; color: white; margin: 0 0 24px 0; font-family: Georgia, serif; box-shadow: 0 4px 14px rgba(11,61,145,0.25);">

<div style="display: flex; justify-content: space-between; align-items: center; margin-bottom: 10px;">
  <span style="font-size: 11px; font-weight: bold; letter-spacing: 2px; color: #BBDEFB;">MEB · YEĞİTEK · DERİN ÖĞRENME SERİSİ</span>
  <span style="font-size: 11px; background: rgba(255,255,255,0.18); padding: 5px 12px; border-radius: 20px; color: white; font-weight: bold;">DERS 4 / 5</span>
</div>

<h1 style="margin: 8px 0 4px 0; color: white; font-size: 28px; line-height: 1.2;">👕 CNN ile Giysi Tanıma</h1>

<p style="margin: 8px 0 0 0; color: #E3F2FD; font-size: 13px;">
<strong>Yenilik ve Eğitim Teknolojileri Genel Müdürlüğü</strong> — Öğretmenler için Uygulamalı Yapay Zeka<br>
<span style="color: #BBDEFB;">⏱️ Süre: ~60 dakika · 🖥️ Ortam: Google Colab (GPU önerilir) veya yerel Python</span>
</p>

<div style="border-top: 1px solid rgba(255,255,255,0.22); margin-top: 14px; padding-top: 12px; font-size: 12px; color: #E3F2FD;">
🎯 Bu notebook <strong>non-teknik kitle</strong> için hazırlanmıştır · Kavramlar günlük hayattan analojilerle anlatılır · Kod minimum, açıklama maksimumdur.
</div>

</div>

## 🎯 Bu Notebook'ta Ne Öğreneceğiz?

Önceki notebook'ta MNIST'te ~%97 doğruluk aldık. Ama o iş "kolay"dı — basit el yazısı rakamları. Şimdi sıra **gerçek görüntülere** geliyor: **giysiler**.

Sonunda şunları bileceksiniz:

- 🖼️ **Konvolüsyon (Convolution)** nedir? Niye görüntülerde **Dense katmandan daha iyidir?**
- 🔍 **Filtre / Kernel** nedir? Görüntüde "ne arar"?
- 🌊 **Pooling** ne işe yarar?
- 👁️ **Feature map** — modelin "gözü" ne görüyor görselleştirelim
- 🧥 Fashion-MNIST ile gerçek bir **giysi tanıma** modeli + Gradio arayüzü

## 🤔 Önce Bir Senaryo

Bir bebek düşünün. Annesini fotoğraftan tanır — saçları, gözleri, gülümsemesi gibi **küçük özelliklere** odaklanır. Ortam değişse, ışık değişse, mesafe değişse bile annesini tanır.

Düz Dense katmanlar bu işi iyi yapamaz çünkü:
- Görüntüyü düzleştirip **piksel piksel** bakar
- Pikseller az bile kayma yapsa "yeni bir şey" gibi görünür
- 28×28 görüntüde 784 girdi, **konum bilgisi kaybolur**

CNN (Konvolüsyonel Sinir Ağı) farklı çalışır:
- Görüntüyü **küçük pencerelerle** tarar
- Her pencerede **kenar, köşe, doku** gibi özellik arar
- Konumdan bağımsız çalışır

> 🎓 **Analoji:** Düz katman = Romanı kelime kelime sırayla okumak. CNN = Önce başlıkları, sonra paragraf özetlerini, sonra detayları okumak.


## 📦 Kurulum

In [ ]:
!pip install -q gradio

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tensorflow import keras
from tensorflow.keras import layers
import gradio as gr

keras.utils.set_random_seed(42)
print(f"✅ Keras {keras.__version__} hazır.")

## 📊 Fashion-MNIST Veri Seti

MNIST gibi 70.000 adet 28×28 görüntü — **ama rakam yerine giysi**! 10 sınıf:

| 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 |
|---|---|---|---|---|---|---|---|---|---|
| Tişört | Pantolon | Kazak | Elbise | Mont | Sandalet | Gömlek | Spor Ayakkabı | Çanta | Bot |


In [ ]:
(X_train, y_train), (X_test, y_test) = keras.datasets.fashion_mnist.load_data()

sinif_isimleri = ['Tişört', 'Pantolon', 'Kazak', 'Elbise', 'Mont',
                   'Sandalet', 'Gömlek', 'Spor Ayakkabı', 'Çanta', 'Bot']

# Önişleme: normalize + boyut ekle (CNN'in 'kanal' boyutu beklediği için)
X_train = X_train.astype('float32') / 255.0
X_test  = X_test.astype('float32')  / 255.0

X_train = X_train.reshape(-1, 28, 28, 1)
X_test  = X_test.reshape(-1, 28, 28, 1)

print(f"✅ Eğitim: {X_train.shape}, Test: {X_test.shape}")

### Bazı Örnekler

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(X_train[i].squeeze(), cmap='gray')
    ax.set_title(sinif_isimleri[y_train[i]], fontsize=11)
    ax.axis('off')
plt.suptitle('👕 Fashion-MNIST Örnekleri', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 🧠 CNN Mimarisi — Adım Adım

Modelimiz şu yapıda olacak:

```
[28x28x1 görüntü]
   ↓ Conv2D(32 filtre, 3x3) → ReLU
[26x26x32]
   ↓ MaxPool(2x2)
[13x13x32]
   ↓ Conv2D(64 filtre, 3x3) → ReLU
[11x11x64]
   ↓ MaxPool(2x2)
[5x5x64]
   ↓ Flatten
[1600]
   ↓ Dense(64, ReLU) → Dropout
[64]
   ↓ Dense(10, Softmax)
[10 olasılık]
```

### Anahtar Kavramlar

#### 🔍 Conv2D — Konvolüsyon Katmanı
- **32 filtre, 3x3:** 32 farklı 3×3'lük "pencere" görüntüyü tarar
- Her filtre **bir özellik arar** (kenar, köşe, çizgi, doku...)
- Filtreler **eğitilirken** otomatik öğrenilir — bizim tasarlamamız gerekmez!

#### 🌊 MaxPooling2D — Havuzlama
- 2×2'lik bölgelere bakar, **en yüksek değeri** alır
- Görüntüyü **küçültür** ama önemli özellikler kalır
- Hesabı azaltır + küçük kayma/dönüşlere dayanıklılık verir

> 🎓 **Pooling'in mantığı:** Bir fotoğrafta köpek mi var diye bakarken her pikseli incelemiyorsun, "bir bölgede köpek özellikleri var mı?" diye bakıyorsun.


In [ ]:
model = keras.Sequential([
    layers.Input(shape=(28, 28, 1)),

    # 1. Konvolüsyon bloğu
    layers.Conv2D(32, kernel_size=(3,3), activation='relu', name='konv_1'),
    layers.MaxPooling2D(pool_size=(2,2), name='havuz_1'),

    # 2. Konvolüsyon bloğu
    layers.Conv2D(64, kernel_size=(3,3), activation='relu', name='konv_2'),
    layers.MaxPooling2D(pool_size=(2,2), name='havuz_2'),

    # Düzleştir + tam bağlı katmanlar
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(10, activation='softmax')
])

model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

model.summary()

## 🚀 Eğitim

CNN'ler hesabı yoğun olduğu için Colab'da **GPU açmanızı öneriyoruz** (Çalışma zamanı → Türü değiştir → GPU).

In [ ]:
gecmis = model.fit(
    X_train, y_train,
    epochs=8,
    batch_size=128,
    validation_split=0.1,
    verbose=1
)

In [ ]:
# Eğitim grafiği
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(gecmis.history['accuracy'], label='Eğitim', linewidth=2)
axes[0].plot(gecmis.history['val_accuracy'], label='Doğrulama', linewidth=2)
axes[0].set_title('Doğruluk', fontweight='bold'); axes[0].legend(); axes[0].set_xlabel('Epoch')

axes[1].plot(gecmis.history['loss'], label='Eğitim', linewidth=2)
axes[1].plot(gecmis.history['val_loss'], label='Doğrulama', linewidth=2)
axes[1].set_title('Hata', fontweight='bold'); axes[1].legend(); axes[1].set_xlabel('Epoch')

plt.tight_layout()
plt.show()

In [ ]:
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"🎯 Test doğruluğu: {test_acc * 100:.2f}%")

## 👁️ Modelin Gözüyle Bakalım — Feature Map'ler

Bu, CNN'in **büyüsünü** gerçekten gösteren kısım! Modelin her katmanda **ne gördüğünü** görselleştireceğiz.

### Nasıl Yapacağız?
İlk konvolüsyon katmanının çıktısını alıp 32 filtreden ilk 8'ini görselleştireceğiz.

In [ ]:
# Sadece ilk konv katmanına kadar olan kısmı içeren mini-model
goz_modeli = keras.Model(inputs=model.input, outputs=model.get_layer('konv_1').output)

# Bir test görüntüsü seç
ornek = X_test[0:1]
ornek_etiket = sinif_isimleri[y_test[0]]

# Bu görüntü için filtre çıktılarını al
feature_maps = goz_modeli.predict(ornek, verbose=0)

print(f"Feature map boyutu: {feature_maps.shape}")
print(f"Yani {feature_maps.shape[-1]} farklı filtre, her biri {feature_maps.shape[1]}x{feature_maps.shape[2]}")

# Görselleştirme
fig, axes = plt.subplots(2, 5, figsize=(13, 5))

# Orijinal görüntü
axes[0,0].imshow(ornek.squeeze(), cmap='gray')
axes[0,0].set_title(f'Orijinal: {ornek_etiket}', fontweight='bold')
axes[0,0].axis('off')

# 9 farklı filtre çıktısı
for i in range(9):
    ax = axes[(i+1)//5, (i+1)%5]
    ax.imshow(feature_maps[0, :, :, i], cmap='viridis')
    ax.set_title(f'Filtre {i+1}', fontsize=10)
    ax.axis('off')

plt.suptitle("🔍 İlk Konvolüsyon Katmanının 'Gördükleri'", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n💡 Her filtre farklı bir özelliğe (kenar, dolgu, doku) tepki veriyor!")

## 💾 Modeli Kaydet

In [ ]:
model.save('giysi_modeli.keras')
print("✅ Model kaydedildi: giysi_modeli.keras")

## 🌐 Gradio ile Giysi Tanıma Arayüzü

Kullanıcı bir resim yükleyecek, model tahmin edecek.

In [ ]:
from PIL import Image as PILImage

def giysi_tahmin_et(resim):
    if resim is None:
        return {sinif: 0.0 for sinif in sinif_isimleri}

    # PIL Image gelir, gri tonlamaya çevir + 28x28 yap
    if isinstance(resim, np.ndarray):
        pil = PILImage.fromarray(resim)
    else:
        pil = resim

    pil = pil.convert('L').resize((28, 28))
    img = np.array(pil).astype('float32') / 255.0

    # Fashion-MNIST'te giysiler beyaz arka plan üzerinde GRİ — tersini de deneyelim
    # Eğer ortalama parlaksa renkleri ters çevir (kullanıcı muhtemelen siyah yazıdı)
    if img.mean() > 0.5:
        img = 1.0 - img

    img = img.reshape(1, 28, 28, 1)
    olasiliklar = model.predict(img, verbose=0)[0]
    return {sinif_isimleri[i]: float(olasiliklar[i]) for i in range(10)}


arayuz = gr.Interface(
    fn=giysi_tahmin_et,
    inputs=gr.Image(type='numpy', label='👕 Bir giysi resmi yükleyin', height=300),
    outputs=gr.Label(num_top_classes=3, label='🎯 En Olası 3 Sınıf'),
    title='👕 MEB YEĞİTEK · CNN ile Giysi Tanıma',
    description=('Fashion-MNIST üzerinde eğitilmiş CNN modeli. '
                 'Resimler **siyah arka plan üzerinde gri/beyaz giysi** şeklinde olduğu için '
                 'kendi resimleriniz beklenildiği gibi sınıflandırılmayabilir. '
                 'En iyi sonuç için Fashion-MNIST görseline benzer resimler kullanın.'),
    flagging_mode='never'
)

arayuz.launch(share=False)

## 📝 Özet

| Terim | Türkçe Karşılığı | Anlamı |
|---|---|---|
| **Convolution** | Konvolüsyon | Filtreyi görüntüde kaydırarak özellik arama |
| **Filter / Kernel** | Filtre / Çekirdek | Belirli bir özellik (kenar, doku) arayan küçük matris |
| **Feature Map** | Özellik Haritası | Filtrenin tüm görüntüde "bulduklarını" gösteren çıktı |
| **Pooling** | Havuzlama | Görüntüyü küçültürken önemli özellikleri tutma |
| **Stride** | Adım | Filtrenin kaç piksel kaydığı |

### 🎓 Anahtar İçgörü

> **CNN, bir görüntüye insan gibi bakar:** önce kenarları, sonra parçaları, sonra bütünü görür.

Bu notebook'ta **hazır veri seti** kullandık. Sıradaki notebook'ta **kendi resimlerinizi** kullanarak özel sınıflandırıcı yapacağız — üstelik **çok daha az veri** ile! 🚀

<div style="background: #0B3D91; color: #BBDEFB; padding: 26px 30px; border-radius: 14px; margin-top: 32px; font-family: Georgia, serif;">

<h3 style="color: white; margin: 0 0 12px 0; border-bottom: 2px solid #FFC107; padding-bottom: 8px; display: inline-block;">MEB · YEĞİTEK</h3>

<p style="font-family: Calibri, sans-serif; font-size: 14px; color: #E3F2FD; margin: 0 0 8px 0; line-height: 1.6;">
<strong>Yenilik ve Eğitim Teknolojileri Genel Müdürlüğü</strong><br>
Öğretmen ve eğitimciler için uygulamalı derin öğrenme eğitim serisi.
</p>

<p style="font-family: Calibri, sans-serif; font-size: 12px; color: #BBDEFB; margin: 12px 0 0 0;">
🎓 Bu notebook eğitim amaçlıdır · Kullanılan tüm kütüphaneler açık kaynaktır.
</p>

<div style="background: rgba(255,255,255,0.08); border-left: 3px solid #FFC107; padding: 12px 16px; margin-top: 16px; font-family: Calibri, sans-serif; font-size: 13px; color: #FFF9E6;">
➡️ <strong>Bir sonraki notebook:</strong> <code>05_kendi_fotograflarinla_egitim.ipynb</code> — Transfer learning ile kendi fotoğraflarınızla profesyonel sınıflandırıcı yapacağız.
</div>
<p style="font-family: Calibri, sans-serif; font-size: 11px; color: #90CAF9; margin: 14px 0 0 0; text-align: center; border-top: 1px solid rgba(255,255,255,0.15); padding-top: 12px;">
© 2026 MEB YEĞİTEK · Derin Öğrenme Serisi
</p>

</div>